# 04 — Dataset Splits Verification

Verifies the 40 / 10 / 10 / 40 split (train / validation / test / production) that notebook 03 wrote to S3 and to the SageMaker Feature Store offline store.

Required for the Week module deliverable:

> Split your feature data into training (~40%), test (~10%), validation (~10%) datasets. Reserve some data for "production data" (~40%).

In [ ]:
import boto3
import pandas as pd
import awswrangler as wr
from pathlib import Path

%store -r bucket
%store -r split_prefix
%store -r feature_group_name

print("Bucket:           ", bucket)
print("Split S3 prefix:  ", split_prefix)
print("Feature group:    ", feature_group_name)

## Read each split CSV back from S3

In [ ]:
splits = {}
for name in ["train", "validation", "test", "production"]:
    path = f"s3://{bucket}/{split_prefix}/{name}/{name}.csv"
    splits[name] = wr.s3.read_csv(path=path)
    print(f"{name:<10} rows={len(splits[name]):>7,}  positive_share={splits[name]['sentiment_label'].mean():.4f}")

## Build the summary table

In [ ]:
total_rows = sum(len(df) for df in splits.values())
rows = []
for name, df in splits.items():
    rows.append({
        "split_type": name,
        "row_count": len(df),
        "share_of_total": round(len(df) / total_rows, 4),
        "positive_count": int((df["sentiment_label"] == 1).sum()),
        "negative_count": int((df["sentiment_label"] == 0).sum()),
        "positive_share": round(df["sentiment_label"].mean(), 4),
        "avg_review_word_count": round(df["review_word_count"].mean(), 1),
    })
summary_df = pd.DataFrame(rows).set_index("split_type").loc[["train", "validation", "test", "production"]]
summary_df

## Verify each class meets the rubric's 10,000 records per class minimum

The training split is what models learn from, so we apply the floor there.

In [ ]:
train_pos = int((splits["train"]["sentiment_label"] == 1).sum())
train_neg = int((splits["train"]["sentiment_label"] == 0).sum())
print(f"Training positive count: {train_pos:,}")
print(f"Training negative count: {train_neg:,}")

rubric_passed = train_pos >= 10000 and train_neg >= 10000
print("Rubric (>=10,000 records per class in train):", "PASS" if rubric_passed else "FAIL")
%store rubric_passed

## Persist the summary table to the reports folder

In [ ]:
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "dataset_splits_summary.md"

lines = ["# Dataset Splits Summary", "", "Split policy: train 40% | validation 10% | test 10% | production 40%", ""]
lines.append(summary_df.to_markdown())
lines.append("")
lines.append(f"Training records per class meet >=10,000 rubric requirement: **{'PASS' if rubric_passed else 'FAIL'}**")
lines.append(f"")
lines.append(f"Feature Group: `{feature_group_name}`")
lines.append(f"S3 prefix:    `s3://{bucket}/{split_prefix}/`")

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", report_path)

## Done

The Week module deliverables are complete:

- Raw data is in S3 (`raw/` prefix).
- Athena database `yelp_db` catalogs raw CSV and processed Parquet.
- EDA notebook produced charts and summary in `reports/`.
- Feature engineering ingested labeled records into the SageMaker Feature Store.
- 40 / 10 / 10 / 40 splits are written to S3 and verified above.

Next modules will plug these splits into model training, the SageMaker Pipeline (CI/CD), Model Registry, Batch Transform, and Model Monitor.